In [1]:
def search(s:str):
    search_data = {
    "capital of france": "The capital of France is Paris.",
    "who wrote hamlet": "Hamlet was written by William Shakespeare.",
    "tallest mountain": "Mount Everest is the tallest mountain on Earth.",
    "speed of light": "The speed of light is approximately 299,792,458 meters per second.",
    "population of japan": "Japan has a population of approximately 123 million people.",
    }
    if s.lower() in search_data:
        return search_data[s]
    else:
        return "not found"

In [2]:
def calc(a,b,op):
    match op:
        case '+':
            return a+b
        case '-':
            return a-b
        case '*':
            return a*b
        case '/':
            return a/b
        case _:
            return 'Unknown operator'

In [6]:
from dotenv import load_dotenv
import anthropic
import os

In [7]:
load_dotenv()
api_key = os.getenv('API_KEY')
client = anthropic.Anthropic(api_key = api_key)

In [11]:
response = client.messages.create(
    model = 'claude-haiku-4-5-20251001',
    max_tokens = 100,
    system = 'You will output in the following structure: First a Thought. Thought will be what your reasoning is what you think should be done. followed by thought, give either an Action or Answer but never both. Action will be what you require to be done like a tool call and specifically mention the exact call for example for search tool, output would look like "Action: search("capital of france")" or for calc tool would be "Action: calc(1,3,"/")". Answer only when you are confident you have the final output ready. Each should have a parseable prefix like "Thought:" for thoughts, "Action:" for actions and "Answer:" for answers. the tools available are "calc" and "search". calc takes 2 integers and an operator. search takes a string.',
    messages = [
        {'role':'user','content':'what is 847/37?'}
    ]
)

In [12]:
print(response.content[0].text)

Thought: I need to calculate 847 divided by 37. I'll use the calc tool to get the exact answer.

Action: calc(847, 37, "/")


In [ ]:
output = response.content[0].text.split('\n')
for i in output:
    if 'Action: ' in i:
        call = i.replace('Action: ','')
        call = call.replace('"','').replace(')','')
        call = call.split('(')
        func_name = call[0]
        if func_name=='calc': 
            args = call[1].replace(' ','').split(',')
        elif func_name=='search':
            args = call[1]
        match func_name:
            case 'calc':
                observation = calc(int(args[0]), int(args[1]), args[2])
            case 'search':
                observation = search(args)



In [ ]:
response = client.messages.create(
    model = 'claude-haiku-4-5-20251001',
    max_tokens = 100,
    system = 'You will output in the following structure: First a Thought. Thought will be what your reasoning is what you think should be done. followed by thought, give either an Action or Answer but never both. Action will be what you require to be done like a tool call and specifically mention the exact call for example for search tool, output would look like "Action: search("capital of france")" or for calc tool would be "Action: calc(1,3,"/")". Answer only when you are confident you have the final output ready. Each should have a parseable prefix like "Thought:" for thoughts, "Action:" for actions and "Answer:" for answers. the tools available are "calc" and "search". calc takes 2 integers and an operator. search takes a string.',
    messages = [
        {'role':'user','content':'what is 847/37?'},
        {'role':'assistant','content':response.content[0].text},
        {'role':'user','content':f'here is the output of that function: {observation}'}
    ]
)

In [ ]:
messages = [{'role':'user','content':'what is 847/37?'}]
while True:
    response = client.messages.create(
    model = 'claude-haiku-4-5-20251001',
    max_tokens = 100,
    system = 'You will output in the following structure: First a Thought. Thought will be what your reasoning is what you think should be done. followed by thought, give either an Action or Answer but never both. Action will be what you require to be done like a tool call and specifically mention the exact call for example for search tool, output would look like "Action: search("capital of france")" or for calc tool would be "Action: calc(1,3,"/")". Answer only when you are confident you have the final output ready. Each should have a parseable prefix like "Thought:" for thoughts, "Action:" for actions and "Answer:" for answers. the tools available are "calc" and "search". calc takes 2 integers and an operator. search takes a string.',
    messages = messages
    )
    output = response.content[0].text.split('\n')
    if any("Answer: " in  i for i in output):
        print(response.content[0].text)
        break
    for i in output:
        if 'Action: ' in i:
            call = i.replace('Action: ','')
            call = call.replace('"','').replace(')','')
            call = call.split('(')
            func_name = call[0]
            if func_name=='calc': 
                args = call[1].replace(' ','').split(',')
            elif func_name=='search':
                args = call[1]
            match func_name:
                case 'calc':
                    observation = calc(int(args[0]), int(args[1]), args[2])
                case 'search':
                    observation = search(args)
                case _:
                    observation = 'tool not found'
    messages.append({'role':'assistant','content':response.content[0].text})
    messages.append({'role':'user','content':f'here is the output of that function: {observation}'})